# HAPI‑FHIR + Synthea Sandbox (PowerShell Edition)

This notebook reproduces the entire setup and validation workflow we walked through in chat, but every shell cell is written for **PowerShell** instead of Git Bash.  Run the cells step‑by‑step in VS Code’s Jupyter view (kernel: *Python* is fine because each shell cell is self‑contained).

## Prerequisites
* Docker Desktop running locally
* Synthea export folder at `D:\synthea_out\fhir` (one **transaction Bundle** JSON per patient)
* A PowerShell‑enabled Jupyter environment (VS Code does this automatically when you prefix cells with `%%powershell`)

## 1  Start (or restart) the HAPI‑FHIR server container


**Option A – PostgreSQL persistence**
Use `docker-compose.yml` to run HAPI-FHIR with PostgreSQL (recommended for large imports)
Use Docker Compose with the provided `docker-compose.yml` to launch HAPI‑FHIR backed by PostgreSQL. This avoids the H2 memory limits when loading many bundles. Run the commands below in a PowerShell terminal:


```powershell
docker compose down          # stop and remove old containers
docker compose pull          # fetch images
docker compose up -d         # start stack
docker compose logs -f hapi  # wait for 'Started Application'
```



**Option B – H2 with on-disk files**
Start with `docker-compose-h2.yml` and restart the container as needed to free memory:


In [21]:
import subprocess, os

# Option B – Restart H2-based HAPI-FHIR container and capture logs
log_dir = r"\\Desktop-family\k\self_learning_healthcarebigdata\02_hapi_fhir_synthea_load_powershell"
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, "hapi_log.txt")

# Stop H2 stack
print("Stopping H2 stack...")
subprocess.run(["docker", "compose", "-f", "docker-compose-h2.yml", "down"], check=True)

# Start H2 stack
print("Starting H2 stack in detached mode...")
subprocess.run(["docker", "compose", "-f", "docker-compose-h2.yml", "up", "-d"], check=True)

# Stream logs to file in background via PowerShell
# Start background streaming of HAPI logs with Tee-Object
ps_script = r"& { docker logs -f hapi-fhir | Tee-Object -FilePath '\\Desktop-family\k\self_learning_healthcarebigdata\02_hapi_fhir_server_setup_powershell\hapi_log.log' }"
proc = subprocess.Popen([
    "powershell", "-NoLogo", "-NoProfile", "-Command", ps_script
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f"Streaming logs in background (PID: {proc.pid}) to the log file")



Stopping H2 stack...
Starting H2 stack in detached mode...
Starting H2 stack in detached mode...
Streaming logs in background (PID: 6412) to the log file
Streaming logs in background (PID: 6412) to the log file


## 2  Some local checkings

### 1. Health check: CapabilityStatement should return JSON

In [23]:
import subprocess

# PowerShell script to check FHIR server health with explicit URL
ps_script = """
$FHIR_BASE = "http://localhost:8080/fhir/"
Invoke-RestMethod -Uri ($FHIR_BASE + "metadata") | ConvertTo-Json -Depth 1 | Select-Object -First 20
"""

# Execute PowerShell script
result = subprocess.run(["powershell", "-Command", ps_script], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

{
    "resourceType":  "CapabilityStatement",
    "id":  "a66d1f8e-c898-4fbc-aee5-b3ebae1b90e9",
    "text":  {
                 "status":  "generated",
                 "div":  "\u003cdiv xmlns=\"http://www.w3.org/1999/xhtml\"\u003eHAPI FHIR Server\u003c/div\u003e"
             },
    "name":  "RestServer",
    "status":  "active",
    "date":  "2025-05-27T13:54:32Z",
    "publisher":  "Not provided",
    "kind":  "instance",
    "software":  {
                     "name":  "HAPI FHIR Server",
                     "version":  "8.2.0"
                 },
    "implementation":  {
                           "description":  "HAPI FHIR R4 Server",
                           "url":  "http://localhost:8080/fhir"
                       },
    "fhirVersion":  "4.0.1",
    "format":  [
                   "application/fhir+xml",
                   "xml",
                   "application/fhir+json",
                   "json",
                   "application/x-turtle",
                   "ttl",
   

### 2. Verify Docker container is running properly


In [26]:
import subprocess

ps_script = """
# Check if HAPI FHIR container is running
Write-Output "Checking Docker container status:"
docker ps -a | Select-String -Pattern "hapi-fhir"

# Check container logs for errors
Write-Output "`nContainer logs (last 20 lines):"
docker logs --tail 20 hapi-fhir
"""

result = subprocess.run(["powershell", "-Command", ps_script], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

Checking Docker container status:

b7dbced82068   hapiproject/hapi:v8.2.0-1   "java --class-path /…"   2 minutes ago   Up 2 minutes   
0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp   hapi-fhir

Container logs (last 20 lines):
2025-07-05T20:02:29.312Z  INFO 1 --- [           main] c.u.f.j.sched.BaseSchedulerServiceImpl   : Scheduling clustered job ca.uhn.fhir.jpa.search.warm.CacheWarmingSvcImpl with interval 00:00:10.000
2025-07-05T20:02:29.314Z  INFO 1 --- [           main] c.u.f.j.sched.BaseSchedulerServiceImpl   : Scheduling clustered job ca.uhn.fhir.jpa.term.TermReadSvcImpl with interval 00:10:00
2025-07-05T20:02:29.314Z  INFO 1 --- [           main] c.u.f.j.sched.BaseSchedulerServiceImpl   : Scheduling local job ca.uhn.fhir.jpa.term.TermReindexingSvcImpl with interval 00:01:00.000
2025-07-05T20:02:29.314Z  INFO 1 --- [           main] c.u.f.j.sched.BaseSchedulerServiceImpl   : Scheduling clustered job ca.uhn.fhir.jpa.search.SearchUrlJobMaintenanceSvcImpl$SearchUrlMaintenanceJob with 

### 3. Test basic connectivity to the FHIR server


In [27]:
import subprocess

ps_script = """
Write-Output "Testing basic connectivity to FHIR server..."
try {
    $response = Invoke-WebRequest -Uri "http://localhost:8080/fhir/metadata" -TimeoutSec 5
    Write-Output "Connection successful! Status code: $($response.StatusCode)"
} catch {
    Write-Output "Failed to connect: $_"
    
    # Check if service is listening on port
    Write-Output "`nChecking port 8080:"
    netstat -ano | findstr :8080
}
"""

result = subprocess.run(["powershell", "-Command", ps_script], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

Testing basic connectivity to FHIR server...
Connection successful! Status code: 200



### 4. Testing if a HAPI FHIR Server is Running Correctly


In [29]:
import subprocess, sys

ps_script = r"""
$fhirBase = "http://localhost:8080/fhir"
$results = [ordered]@{}

Write-Output "========== HAPI FHIR SERVER HEALTH CHECK =========="

# 1. Basic connectivity test
Write-Output "`n1. Testing basic connectivity..."
try {
    $response = Invoke-WebRequest -Uri "$fhirBase/metadata" -TimeoutSec 5 -UseBasicParsing
    $results["Basic Connectivity"] = "OK (Status: $($response.StatusCode))"
    Write-Output "Server is reachable! Status code: $($response.StatusCode)"
} catch {
    $results["Basic Connectivity"] = "FAILED: $($_.Exception.Message)"
    Write-Output "Failed to connect: $($_.Exception.Message)"
}
"""

print("Running comprehensive HAPI FHIR server health check...")
result = subprocess.run(["powershell", "-NoLogo", "-NoProfile", "-Command", ps_script], 
                        capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr, file=sys.stderr)

Running comprehensive HAPI FHIR server health check...
========== HAPI FHIR SERVER HEALTH CHECK ==========

1. Testing basic connectivity...
Server is reachable! Status code: 200



### 5. Check server version and capabilities

In [ ]:


import subprocess, sys

ps_script = r"""
$fhirBase = "http://localhost:8080/fhir"
$results = [ordered]@{}

Write-Output "========== HAPI FHIR SERVER HEALTH CHECK =========="
try {
    $metadata = Invoke-RestMethod -Uri "$fhirBase/metadata" -TimeoutSec 10
    $version = $metadata.software.version
    $fhirVersion = $metadata.fhirVersion
    $results["Server Version"] = $version
    $results["FHIR Version"] = $fhirVersion
    
    Write-Output "   Server info: HAPI FHIR $version (FHIR $fhirVersion)"
    
    # List supported resource types
    $resourceTypes = $metadata.rest.resource.type | Sort-Object -Unique
    $results["Resource Types"] = "$($resourceTypes.Count) types supported"
    Write-Output "   Supports $($resourceTypes.Count) resource types"
    
    # Check if common resources are supported
    $commonTypes = @("Patient", "Observation", "Encounter", "Condition")
    foreach ($type in $commonTypes) {
        if ($resourceTypes -contains $type) {
            Write-Output "   Supports $type resources"
        } else {
            Write-Output "Does NOT support $type resources"
        }
    }
    
    # List operations
    $operations = $metadata.rest.operation.name | Sort-Object -Unique
    if ($operations) {
        $results["Operations"] = "$($operations.Count) operations supported"
        Write-Output "   Supports $($operations.Count) operations: $($operations -join ', ')"
    } else {
        $results["Operations"] = "None found"
        Write-Output "No operations found in capabilities statement"
    }
    
} catch {
    $results["Server Capabilities"] = " FAILED: $($_.Exception.Message)"
    Write-Output "   Failed to retrieve capabilities: $($_.Exception.Message)"
}
"""

print("Running comprehensive HAPI FHIR server health check...")
result = subprocess.run(["powershell", "-NoLogo", "-NoProfile", "-Command", ps_script], 
                        capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr, file=sys.stderr)
    


Running comprehensive HAPI FHIR server health check...
========== HAPI FHIR SERVER HEALTH CHECK ==========
   Server info: HAPI FHIR 8.2.0 (FHIR 4.0.1)
   Supports 146 resource types
   Supports Patient resources
   Supports Observation resources
   Supports Encounter resources
   Supports Condition resources
   Supports 9 operations: diff, expunge, get-resource-counts, hapi.fhir.replace-references, mark-all-resources-for-reindexing, meta, perform-reindexing-pass, reindex, reindex-terminology

